In [31]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import random
import os
import math

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ======================================================
# [V14] 14.08m 벽 돌파 - 문제점 수정
# ======================================================
BATCH_SIZE = 64
LR_BASE = 1e-3
WARMUP_EPOCHS = 3
EPOCHS_BASE = 50
DROPOUT = 0.2
MAX_SEQ_LEN = 30
GRAD_CLIP = 1.0

HIDDEN_DIM = 256
LSTM_LAYERS = 3
BIDIRECTIONAL = True

# MDN 파라미터
NUM_GAUSSIANS = 2
MIN_SIGMA = 0.05
MAX_SIGMA = 0.25

# 🔧 수정1: MSE 가중치를 충분히 높게 유지
CURRICULUM_START_MSE_WEIGHT = 1.5
CURRICULUM_END_MSE_WEIGHT = 1.0  # 🔧 0.3 → 1.0: 거리 맞추기를 1순위로 유지
CURRICULUM_TRANSITION_EPOCH = 40  # 🔧 25 → 40: 천천히 전환

# 🔧 수정2: 엔트로피 정규화 대폭 강화
ENTROPY_WEIGHT = 0.5  # 🔧 0.01 → 0.5: 50배 증가로 강력한 제약
MIN_ENTROPY_TARGET = 0.5

# 정보 병목 파라미터
BOTTLENECK_DIM_RATIO = 0.5

# 🔧 수정3: Early Stopping 파라미터 추가
EARLY_STOPPING_PATIENCE = 10  # Val Distance가 10 에포크 동안 개선 안 되면 멈춤

print(f"\n[V14] 14.08m 벽 돌파 - 문제점 수정")
print(f"="*70)
print(f"🔧 핵심 수정사항:")
print(f"  1. MSE 가중치 하한선 상향 (0.3 → 1.0)")
print(f"     - 거리 맞추기를 끝까지 1순위로 유지")
print(f"  2. 엔트로피 정규화 강화 (0.01 → 0.5, 50배)")
print(f"     - 과도한 확신에 강력한 벌점")
print(f"  3. Early Stopping 추가 (patience={EARLY_STOPPING_PATIENCE})")
print(f"     - Val Distance 기준으로 조기 종료")
print(f"  4. 커리큘럼 전환 지연 (epoch 25 → 40)")
print(f"="*70)

Device: cuda

[V14] 14.08m 벽 돌파 - 문제점 수정
🔧 핵심 수정사항:
  1. MSE 가중치 하한선 상향 (0.3 → 1.0)
     - 거리 맞추기를 끝까지 1순위로 유지
  2. 엔트로피 정규화 강화 (0.01 → 0.5, 50배)
     - 과도한 확신에 강력한 벌점
  3. Early Stopping 추가 (patience=10)
     - Val Distance 기준으로 조기 종료
  4. 커리큘럼 전환 지연 (epoch 25 → 40)


In [32]:
# ======================================================
# 데이터 로드 (증강은 나중에!)
# ======================================================
BASE_DIR = "./open_track1"
if not os.path.exists(BASE_DIR): BASE_DIR = "."

TRAIN_PATH = os.path.join(BASE_DIR, "train.csv")
TEST_META_PATH = os.path.join(BASE_DIR, "test.csv")
MATCH_PATH = os.path.join(BASE_DIR, "match_info.csv")

train_df = pd.read_csv(TRAIN_PATH)
print(f"✅ Train Loaded: {train_df.shape}")

if os.path.exists(TEST_META_PATH):
    test_meta = pd.read_csv(TEST_META_PATH)
    print(f"ℹ️ Reading {len(test_meta)} test files...")
    test_dfs = []
    for _, row in tqdm(test_meta.iterrows(), total=len(test_meta), desc="Loading Test CSVs"):
        rel_path = row['path']
        paths_to_try = [
            rel_path,
            os.path.join(BASE_DIR, rel_path.lstrip("./")),
            os.path.join(BASE_DIR, "test", str(row['game_id']), os.path.basename(rel_path))
        ]
        for p in paths_to_try:
            if os.path.exists(p):
                test_dfs.append(pd.read_csv(p))
                break
    if test_dfs:
        test_df = pd.concat(test_dfs, ignore_index=True)
        print(f"✅ Test Data Merged: {test_df.shape}")
else:
    raise FileNotFoundError("test.csv not found")

if os.path.exists(MATCH_PATH):
    match_info = pd.read_csv(MATCH_PATH)
    match_subset = match_info[['game_id', 'home_team_id', 'away_team_id', 'home_score', 'away_score', 'venue']]
    train_df = pd.merge(train_df, match_subset, on='game_id', how='left')
    test_df = pd.merge(test_df, match_subset, on='game_id', how='left')

def preprocess(df):
    if 'home_team_id' in df.columns:
        df['is_home'] = (df['team_id'] == df['home_team_id']).astype(float)
    else:
        df['is_home'] = 0.5

    # 🆕 점수 차이 계산 (현재 팀 관점에서)
    if 'home_score' in df.columns and 'away_score' in df.columns:
        # is_home이 1이면 home_score - away_score, 0이면 away_score - home_score
        df['score_diff'] = np.where(
            df['is_home'] == 1.0,
            df['home_score'] - df['away_score'],
            df['away_score'] - df['home_score']
        ).astype(float)
    else:
        df['score_diff'] = 0.0

    # 🆕 남은 시간 계산 (90분 기준)
    if 'period_id' in df.columns and 'time_seconds' in df.columns:
        # period_id=1: 전반, period_id=2: 후반
        # 전반 45분(2700초), 후반 45분(2700초) 총 90분
        elapsed_time = (df['period_id'] - 1) * 2700 + df['time_seconds']
        df['time_remaining'] = (5400 - elapsed_time).clip(lower=0) / 5400.0  # 정규화 (0~1)
    else:
        df['time_remaining'] = 0.5

    if 'end_x' not in df.columns:
        df['end_x'] = 0.0
        df['end_y'] = 0.0
    else:
        df['end_x'] = df['end_x'].fillna(0.0)
        df['end_y'] = df['end_y'].fillna(0.0)
    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

ID_COL = 'game_episode' if 'game_episode' in train_df.columns else 'episode_id'
print(f"\nData Ready. ID Column: {ID_COL}")
print(f"🆕 경기 맥락 피처 추가:")
print(f"  - score_diff: 점수 차이 (현재 팀 관점)")
print(f"  - time_remaining: 남은 시간 비율 (0~1)")
print(f"  - is_home: 홈/원정 (기존)")
print(f"⚠️ 데이터 증강은 train/val split 후에 적용됩니다 (데이터 유출 방지)")

✅ Train Loaded: (356721, 15)
ℹ️ Reading 2414 test files...


Loading Test CSVs: 100%|██████████| 2414/2414 [00:04<00:00, 539.06it/s]


✅ Test Data Merged: (53110, 15)

Data Ready. ID Column: game_episode
🆕 경기 맥락 피처 추가:
  - score_diff: 점수 차이 (현재 팀 관점)
  - time_remaining: 남은 시간 비율 (0~1)
  - is_home: 홈/원정 (기존)
⚠️ 데이터 증강은 train/val split 후에 적용됩니다 (데이터 유출 방지)


In [33]:
# ======================================================
# [V15] 피처 엔지니어링 + N-gram (type + result 조합)
# ======================================================
TOP_TYPES = ['Pass', 'Carry', 'Recovery', 'Interception', 'Duel', 'Tackle', 
             'Throw-In', 'Clearance', 'Intervention', 'Block', 'Pass_Freekick', 
             'Cross', 'Goal Kick', 'Error', 'Shot']
ALL_RESULTS = ['Successful', 'Unsuccessful', 'On Target', 'Yellow_Card', 
               'Blocked', 'Keeper Rush-Out', 'Low Quality Shot', 'Off Target']

# 🆕 N-gram 패턴 (type + result 조합)
TOP_3GRAMS = []
TOP_5GRAMS = []
NGRAM_3_SIZE = 20
NGRAM_5_SIZE = 20

# 패턴 to index 매핑 (Embedding용)
PATTERN_3_TO_IDX = {}
PATTERN_5_TO_IDX = {}

def extract_ngrams_from_data(df, n_size=3, top_k=20):
    """
    type_name + result_name 조합으로 N-gram 패턴 추출
    
    예: "Pass_Successful" -> "Carry_Successful" -> "Pass_Successful"
    """
    from collections import Counter
    patterns = []

    for _, group in df.groupby(ID_COL, sort=False):
        # type + result 조합
        combined = [
            f"{t}_{r}" if pd.notna(r) and r else t
            for t, r in zip(group['type_name'].values, group['result_name'].values)
        ]
        
        for i in range(n_size - 1, len(combined)):
            pattern = tuple(combined[i - n_size + 1 : i + 1])
            patterns.append(pattern)

    # 빈도 계산 및 상위 K개 추출
    counter = Counter(patterns)
    top_patterns = [p for p, _ in counter.most_common(top_k)]
    
    print(f"\n🔍 {n_size}-gram 패턴 분석 (type+result):")
    print(f"   전체 유니크 패턴: {len(counter)}")
    print(f"   상위 {top_k}개:")
    for i, (pattern, count) in enumerate(counter.most_common(min(10, top_k)), 1):
        pattern_str = ' → '.join(pattern)
        print(f"      {i}. {pattern_str}: {count:,}회")
    
    return top_patterns


def make_features(group):
    """
    피처 생성 + N-gram 인덱스 반환
    """
    n = len(group)
    sx = group['start_x'].values / 105.0
    sy = group['start_y'].values / 68.0
    ex = group['end_x'].values / 105.0
    ey = group['end_y'].values / 68.0
    is_home = group['is_home'].values
    
    # 경기 맥락 피처
    score_diff = group['score_diff'].values if 'score_diff' in group.columns else np.zeros(n)
    time_remaining = group['time_remaining'].values if 'time_remaining' in group.columns else np.ones(n) * 0.5
    
    if 'time_seconds' in group.columns:
        times = group['time_seconds'].values
        dt = np.zeros(n, dtype=np.float32)
        dt[1:] = times[1:] - times[:-1]
        dt = np.maximum(dt, 0.1)
    else:
        dt = np.ones(n, dtype=np.float32)

    dx = ex - sx
    dy = ey - sy
    dist_meter = np.sqrt((dx*105)**2 + (dy*68)**2)
    cumsum_dx = np.cumsum(dx) / 105.0
    cumsum_dy = np.cumsum(dy) / 68.0
    lag_dist_m = np.roll(dist_meter, 1); lag_dist_m[0] = 0
    lag_cumsum_dx = np.roll(cumsum_dx, 1); lag_cumsum_dx[0] = 0
    lag_cumsum_dy = np.roll(cumsum_dy, 1); lag_cumsum_dy[0] = 0
    lag_dt = np.roll(dt, 1); lag_dt[0] = 1.0
    lag_speed = lag_dist_m / np.maximum(lag_dt, 0.1)
    
    if 'player_id' in group.columns:
        p_ids = group['player_id'].values
        is_same = np.zeros(n, dtype=np.float32)
        is_same[1:] = (p_ids[1:] == p_ids[:-1]).astype(np.float32)
    else:
        is_same = np.zeros(n, dtype=np.float32)

    progress = np.arange(n) / max(n-1, 1)
    is_second_half = (group['period_id'].values > 1).astype(np.float32) if 'period_id' in group.columns else np.zeros(n)
    
    GOAL_X, GOAL_Y = 105.0, 34.0
    sx_real, sy_real = sx * 105.0, sy * 68.0
    dist_to_goal = np.sqrt((sx_real - GOAL_X)**2 + (sy_real - GOAL_Y)**2) / 105.0
    angle_to_goal = np.arctan2(GOAL_Y - sy_real, GOAL_X - sx_real)
    angle_sin, angle_cos = np.sin(angle_to_goal), np.cos(angle_to_goal)
    dist_to_sideline = np.minimum(sy_real, 68.0 - sy_real) / 68.0
    dist_to_endline = np.minimum(sx_real, 105.0 - sx_real) / 105.0
    
    def get_zone(x_norm):
        if x_norm < 35.0/105.0: return 0
        elif x_norm < 70.0/105.0: return 1
        else: return 2
    
    zones = np.array([get_zone(x) for x in sx])
    zone_onehot = np.zeros((n, 3), dtype=np.float32)
    for i, z in enumerate(zones): zone_onehot[i, z] = 1.0
    
    types_onehot = np.zeros((n, len(TOP_TYPES) + 1), dtype=np.float32)
    for i, t in enumerate(group['type_name'].values):
        types_onehot[i, TOP_TYPES.index(t) if t in TOP_TYPES else -1] = 1.0
    
    results_onehot = np.zeros((n, len(ALL_RESULTS) + 1), dtype=np.float32)
    for i, r in enumerate(group['result_name'].values):
        results_onehot[i, ALL_RESULTS.index(r) if r in ALL_RESULTS else -1] = 1.0

    # 🆕 N-gram 인덱스 생성 (Embedding용)
    combined_list = [
        f"{t}_{r}" if pd.notna(r) and r else t
        for t, r in zip(group['type_name'].values, group['result_name'].values)
    ]
    
    ngram3_idx = np.zeros(n, dtype=np.int64)  # 0 = padding
    ngram5_idx = np.zeros(n, dtype=np.int64)
    
    for i in range(n):
        # 3-gram
        if i >= 2:
            pattern = tuple(combined_list[i-2:i+1])
            if pattern in PATTERN_3_TO_IDX:
                ngram3_idx[i] = PATTERN_3_TO_IDX[pattern]
            else:
                ngram3_idx[i] = len(TOP_3GRAMS) + 1  # 'Others' 인덱스
        
        # 5-gram
        if i >= 4:
            pattern = tuple(combined_list[i-4:i+1])
            if pattern in PATTERN_5_TO_IDX:
                ngram5_idx[i] = PATTERN_5_TO_IDX[pattern]
            else:
                ngram5_idx[i] = len(TOP_5GRAMS) + 1  # 'Others' 인덱스

    # 기본 피처 + N-gram 인덱스
    features = []
    ngram3_indices = []
    ngram5_indices = []
    
    for i in range(n):
        scalars = [sx[i], sy[i], lag_cumsum_dx[i], lag_cumsum_dy[i], lag_dist_m[i]/100.0,
                   lag_speed[i]/10.0, dt[i]/10.0, progress[i], is_home[i], is_same[i],
                   is_second_half[i], dist_to_goal[i], angle_sin[i], angle_cos[i],
                   dist_to_sideline[i], dist_to_endline[i], 
                   score_diff[i]/5.0, time_remaining[i]]
        feat_vec = np.concatenate([scalars, zone_onehot[i], types_onehot[i], results_onehot[i]])
        features.append(feat_vec)
        ngram3_indices.append(ngram3_idx[i])
        ngram5_indices.append(ngram5_idx[i])
        
        if i < n - 1:
            ex_real, ey_real = ex[i] * 105.0, ey[i] * 68.0
            end_dist_to_goal = np.sqrt((ex_real - GOAL_X)**2 + (ey_real - GOAL_Y)**2) / 105.0
            end_angle = np.arctan2(GOAL_Y - ey_real, GOAL_X - ex_real)
            scalars_end = scalars.copy()
            scalars_end[0:2] = [ex[i], ey[i]]
            scalars_end[2:4] = [cumsum_dx[i], cumsum_dy[i]]
            scalars_end[11:16] = [end_dist_to_goal, np.sin(end_angle), np.cos(end_angle),
                                   min(ey_real, 68.0 - ey_real) / 68.0,
                                   min(ex_real, 105.0 - ex_real) / 105.0]
            end_zone_onehot = np.zeros(3, dtype=np.float32)
            end_zone_onehot[get_zone(ex[i])] = 1.0
            feat_vec_end = np.concatenate([scalars_end, end_zone_onehot, types_onehot[i], results_onehot[i]])
            features.append(feat_vec_end)
            ngram3_indices.append(ngram3_idx[i])
            ngram5_indices.append(ngram5_idx[i])
            
    return (np.array(features, dtype=np.float32), 
            np.array(ngram3_indices, dtype=np.int64),
            np.array(ngram5_indices, dtype=np.int64))


# 🆕 N-gram 패턴 추출
print("\n" + "="*70)
print("🔍 N-gram 패턴 추출 중 (type+result 조합)...")
print("="*70)

TOP_3GRAMS = extract_ngrams_from_data(train_df, n_size=3, top_k=NGRAM_3_SIZE)
TOP_5GRAMS = extract_ngrams_from_data(train_df, n_size=5, top_k=NGRAM_5_SIZE)

# 인덱스 매핑 생성 (1부터 시작, 0은 padding)
PATTERN_3_TO_IDX = {pattern: idx + 1 for idx, pattern in enumerate(TOP_3GRAMS)}
PATTERN_5_TO_IDX = {pattern: idx + 1 for idx, pattern in enumerate(TOP_5GRAMS)}

print(f"\n✅ N-gram 패턴 추출 완료:")
print(f"   3-gram: {len(TOP_3GRAMS)}개 (+ Others)")
print(f"   5-gram: {len(TOP_5GRAMS)}개 (+ Others)")
print(f"   Embedding vocab size: 3-gram={len(TOP_3GRAMS)+2}, 5-gram={len(TOP_5GRAMS)+2}")
print(f"   (0=padding, 1~{len(TOP_3GRAMS)}=top patterns, {len(TOP_3GRAMS)+1}=others)")
print("="*70)

# INPUT_DIM 계산
dummy_group = train_df.iloc[:5].copy()
dummy_feats, _, _ = make_features(dummy_group)
INPUT_DIM = dummy_feats.shape[1]
print(f"\n✅ Base Input Dimension: {INPUT_DIM}")
print(f"🆕 N-gram은 별도 Embedding Layer로 처리 (모델에서 concat)")


🔍 N-gram 패턴 추출 중 (type+result 조합)...

🔍 3-gram 패턴 분석 (type+result):
   전체 유니크 패턴: 1584
   상위 20개:
      1. Pass_Successful → Carry → Pass_Successful: 50,130회
      2. Pass_Successful → Pass_Successful → Pass_Successful: 30,028회
      3. Carry → Pass_Successful → Carry: 27,832회
      4. Pass_Successful → Pass_Successful → Carry: 26,015회
      5. Carry → Pass_Successful → Pass_Successful: 25,251회
      6. Recovery → Carry → Pass_Successful: 6,285회
      7. Pass_Successful → Carry → Pass_Unsuccessful: 6,163회
      8. Recovery → Pass_Successful → Pass_Successful: 5,140회
      9. Recovery → Pass_Successful → Carry: 4,248회
      10. Interception → Clearance → Recovery: 3,942회

🔍 5-gram 패턴 분석 (type+result):
   전체 유니크 패턴: 12084
   상위 20개:
      1. Pass_Successful → Carry → Pass_Successful → Carry → Pass_Successful: 18,246회
      2. Carry → Pass_Successful → Carry → Pass_Successful → Carry: 10,077회
      3. Pass_Successful → Pass_Successful → Pass_Successful → Carry → Pass_Successful: 9,696회
 

In [34]:
# ======================================================
# 데이터셋 (N-gram 인덱스 포함)
# ======================================================
class SoccerDataset(Dataset):
    def __init__(self, df, mode='train', augment_y=False):
        """
        augment_y: Y축 반전 증강 여부
        """
        self.mode = mode
        self.augment_y = augment_y
        self.episodes = []
        self.ngram3_indices = []
        self.ngram5_indices = []
        self.targets = []
        self.team_ids = []
        self.episode_ids = []
        
        for name, group in tqdm(df.groupby(ID_COL, sort=False), desc=f"Dataset ({mode})"):
            if mode == 'train' and len(group) < 2: continue
            
            # 원본 추가
            seq, ng3_idx, ng5_idx = make_features(group)
            team_id = group.iloc[0]['team_id']
            
            if mode == 'train':
                last = group.iloc[-1]
                self.targets.append([last['end_x']/105.0, last['end_y']/68.0])
                self.episodes.append(seq)
                self.ngram3_indices.append(ng3_idx)
                self.ngram5_indices.append(ng5_idx)
                self.team_ids.append(team_id)
            else:
                self.episodes.append(seq)
                self.ngram3_indices.append(ng3_idx)
                self.ngram5_indices.append(ng5_idx)
                self.team_ids.append(team_id)
                self.episode_ids.append(str(name))
            
            # 🆕 Y축 증강 (train만!)
            if mode == 'train' and augment_y:
                group_aug = group.copy()
                group_aug['start_y'] = 68.0 - group_aug['start_y']
                group_aug['end_y'] = 68.0 - group_aug['end_y']
                
                seq_aug, ng3_idx_aug, ng5_idx_aug = make_features(group_aug)
                last_aug = group_aug.iloc[-1]
                self.targets.append([last_aug['end_x']/105.0, last_aug['end_y']/68.0])
                self.episodes.append(seq_aug)
                self.ngram3_indices.append(ng3_idx_aug)
                self.ngram5_indices.append(ng5_idx_aug)
                self.team_ids.append(team_id)

    def __len__(self): return len(self.episodes)
    
    def __getitem__(self, idx):
        seq = torch.FloatTensor(self.episodes[idx])
        ng3 = torch.LongTensor(self.ngram3_indices[idx])
        ng5 = torch.LongTensor(self.ngram5_indices[idx])
        
        if len(seq) > MAX_SEQ_LEN:
            seq = seq[-MAX_SEQ_LEN:]
            ng3 = ng3[-MAX_SEQ_LEN:]
            ng5 = ng5[-MAX_SEQ_LEN:]
        
        if self.mode == 'train':
            return seq, ng3, ng5, torch.FloatTensor(self.targets[idx]), self.team_ids[idx]
        return seq, ng3, ng5, self.team_ids[idx], self.episode_ids[idx]


def collate_fn(batch):
    """
    Collate function for N-gram support
    """
    seqs = [b[0] for b in batch]
    ng3s = [b[1] for b in batch]
    ng5s = [b[2] for b in batch]
    
    lens = torch.LongTensor([len(s) for s in seqs])
    
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    ng3_padded = pad_sequence(ng3s, batch_first=True, padding_value=0)
    ng5_padded = pad_sequence(ng5s, batch_first=True, padding_value=0)
    
    mask = torch.arange(padded.size(1))[None, :] >= lens[:, None]
    
    # Test mode
    if isinstance(batch[0][4], str):
        return (padded, ng3_padded, ng5_padded, mask, lens, 
                torch.LongTensor([b[3] for b in batch]), 
                [b[4] for b in batch])
    
    # Train mode
    return (padded, ng3_padded, ng5_padded, 
            torch.stack([b[3] for b in batch]), 
            mask, lens, 
            torch.LongTensor([b[4] for b in batch]))


# 데이터셋 생성
full_dataset = SoccerDataset(train_df, mode='train', augment_y=False)
test_dataset = SoccerDataset(test_df, mode='test', augment_y=False)
print(f"✅ Dataset: {len(full_dataset)} episodes (증강 전)")
print(f"✅ Test: {len(test_dataset)} episodes")
print(f"🆕 각 에피소드에 N-gram 인덱스 포함")

Dataset (test): 100%|██████████| 2414/2414 [00:03<00:00, 783.17it/s]

✅ Dataset: 15428 episodes (증강 전)
✅ Test: 2414 episodes
🆕 각 에피소드에 N-gram 인덱스 포함


In [35]:
# ======================================================
# [V15] N-gram Embedding 추가 모델
# ======================================================

# N-gram Embedding 설정
NGRAM_EMBED_DIM = 8  # 각 패턴을 8차원 벡터로 표현


class GeometricEncoder(nn.Module):
    """
    🆕 기하학적 대칭성 인코더
    """
    def __init__(self, output_dim):
        super().__init__()
        self.polar_encoder = nn.Sequential(
            nn.Linear(4, output_dim),
            nn.ReLU(),
            nn.LayerNorm(output_dim)
        )
        
    def forward(self, x):
        pos_x = x[..., 0]
        pos_y = x[..., 1]
        
        goal_x, goal_y = 1.0, 0.5
        dx = pos_x - goal_x
        dy = pos_y - goal_y
        r = torch.sqrt(dx**2 + dy**2)
        theta = torch.atan2(dy, dx)
        
        y_sym = torch.abs(pos_y - 0.5)
        geom_feats = torch.stack([r, torch.sin(theta), torch.cos(theta), y_sym], dim=-1)
        
        return self.polar_encoder(geom_feats)


class InformationBottleneck(nn.Module):
    """정보 병목 레이어"""
    def __init__(self, dim, bottleneck_ratio=0.5, dropout=0.1):
        super().__init__()
        bottleneck_dim = int(dim * bottleneck_ratio)
        
        self.compress = nn.Sequential(
            nn.Linear(dim, bottleneck_dim),
            nn.LayerNorm(bottleneck_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.expand = nn.Sequential(
            nn.Linear(bottleneck_dim, dim),
            nn.LayerNorm(dim)
        )
        
    def forward(self, x):
        compressed = self.compress(x)
        reconstructed = self.expand(compressed)
        return reconstructed


class DeltaOperator(nn.Module):
    """Deep Delta Learning Operator"""
    def __init__(self, dim, dropout=0.1, bottleneck_ratio=0.5):
        super().__init__()
        self.dim = dim
        self.bottleneck = InformationBottleneck(dim, bottleneck_ratio, dropout)
        
        self.k_net = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 2, dim)
        )
        
        self.beta_net = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim // 4, 1),
            nn.Sigmoid()
        )
        
        self.v_net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim)
        )
        
        nn.init.xavier_uniform_(self.k_net[-1].weight, gain=0.1)
        nn.init.xavier_uniform_(self.v_net[0].weight, gain=0.5)
        nn.init.constant_(self.beta_net[-2].bias, -2.0)
    
    def forward(self, x):
        x_filtered = self.bottleneck(x)
        k = self.k_net(x_filtered)
        k = k / (torch.norm(k, dim=-1, keepdim=True) + 1e-8)
        beta = self.beta_net(x_filtered)
        v = self.v_net(x_filtered)
        
        v_T = (v * k).sum(dim=-1, keepdim=True)
        k_T_x = (k * x).sum(dim=-1, keepdim=True)
        output = x + beta * k * (v_T - k_T_x)
        
        return output


class SimpleDeltaBlock(nn.Module):
    def __init__(self, dim, dropout=0.1, bottleneck_ratio=0.5):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.linear = nn.Linear(dim, dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.delta_op = DeltaOperator(dim, dropout, bottleneck_ratio)
    
    def forward(self, x):
        h = self.norm(x)
        h = self.linear(h)
        h = self.activation(h)
        h = self.dropout(h)
        out = x + self.delta_op(h)
        return out


class SpatialAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.goal_attn = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))
        self.zone_attn = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1))
        self.pos_attn = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
        self.fusion = nn.Linear(3, 1)
    def forward(self, x):
        return self.fusion(torch.cat([self.pos_attn(x[..., 0:2]), 
                                       self.goal_attn(x[..., 11:14]), 
                                       self.zone_attn(x[..., 16:19])], dim=-1))


class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, 100, hidden_dim) * 0.02)
        self.temporal_attn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), 
                                           nn.Tanh(), nn.Dropout(0.1), 
                                           nn.Linear(hidden_dim // 2, 1))
    def forward(self, lstm_out):
        return self.temporal_attn(lstm_out + self.pos_encoding[:, :lstm_out.size(1), :])


class SpatialTemporalFusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.spatial_weight = nn.Parameter(torch.tensor(0.5))
        self.temporal_weight = nn.Parameter(torch.tensor(0.5))
        self.combine = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
    def forward(self, s, t):
        return (torch.sigmoid(self.spatial_weight) * s + 
                torch.sigmoid(self.temporal_weight) * t) * self.combine(torch.cat([s, t], -1))


class ImprovedMDNPredictor(nn.Module):
    """
    [V15] N-gram Embedding 추가 모델
    
    🆕 핵심 추가:
    1. 3-gram, 5-gram Embedding Layers
    2. 패턴 간 유사도 학습 가능
    3. type+result 조합으로 풍부한 전술 정보 포착
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_gaussians=2, 
                 bidirectional=True, num_delta_blocks=6, bottleneck_ratio=0.5,
                 ngram3_vocab_size=22, ngram5_vocab_size=22, ngram_embed_dim=8):
        super().__init__()
        self.num_gaussians = num_gaussians
        
        # 🆕 N-gram Embeddings
        self.ngram3_embed = nn.Embedding(ngram3_vocab_size, ngram_embed_dim, padding_idx=0)
        self.ngram5_embed = nn.Embedding(ngram5_vocab_size, ngram_embed_dim, padding_idx=0)
        
        # Geometric encoder
        geom_dim = hidden_dim // 4
        self.geom_encoder = GeometricEncoder(geom_dim)
        
        # Input projection (원본 + geometric + ngrams)
        total_input_dim = input_dim + ngram_embed_dim * 2  # 3-gram + 5-gram
        self.input_proj = nn.Linear(total_input_dim, hidden_dim - geom_dim)
        
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        # DDL with information bottleneck
        self.delta_blocks = nn.ModuleList([
            SimpleDeltaBlock(lstm_output_dim, dropout, bottleneck_ratio) 
            for _ in range(num_delta_blocks)
        ])
        
        self.spatial_attn = SpatialAttention(lstm_output_dim)
        self.temporal_attn = TemporalAttention(lstm_output_dim)
        self.fusion = SpatialTemporalFusion(lstm_output_dim)
        
        # MDN Heads
        self.pi_head = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_output_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_output_dim // 2, num_gaussians)
        )
        self.mu_head = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_output_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_output_dim // 2, num_gaussians * 2)
        )
        self.sigma_head = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_output_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_output_dim // 2, num_gaussians * 2)
        )
        
        nn.init.constant_(self.mu_head[-1].bias, 0.5)
        nn.init.xavier_uniform_(self.mu_head[-1].weight, gain=0.1)

    def forward(self, x, ngram3_idx, ngram5_idx, mask=None, lengths=None):
        batch_size, seq_len = x.size(0), x.size(1)
        
        # 🆕 N-gram embeddings
        ng3_emb = self.ngram3_embed(ngram3_idx)  # (batch, seq_len, embed_dim)
        ng5_emb = self.ngram5_embed(ngram5_idx)
        
        # Geometric encoding
        geom_feats = self.geom_encoder(x)
        
        # Combine: base features + N-gram embeddings
        x_with_ngrams = torch.cat([x, ng3_emb, ng5_emb], dim=-1)
        x_proj = self.input_proj(x_with_ngrams)
        x_combined = torch.cat([x_proj, geom_feats], dim=-1)
        
        # LSTM
        if lengths is not None:
            packed = pack_padded_sequence(x_combined, lengths.cpu(), batch_first=True, enforce_sorted=False)
            lstm_out, _ = self.lstm(packed)
            lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True, total_length=seq_len)
        else:
            lstm_out, _ = self.lstm(x_combined)
        
        # DDL
        for delta_block in self.delta_blocks:
            batch_size, seq_len, hidden_dim = lstm_out.shape
            lstm_out_flat = lstm_out.reshape(-1, hidden_dim)
            lstm_out_flat = delta_block(lstm_out_flat)
            lstm_out = lstm_out_flat.reshape(batch_size, seq_len, hidden_dim)
        
        # Attention
        fused_attn = self.fusion(self.spatial_attn(x), self.temporal_attn(lstm_out))
        if mask is not None:
            fused_attn = fused_attn.masked_fill(mask.unsqueeze(-1), float('-inf'))
        final_attn = torch.softmax(fused_attn, dim=1)
        context = torch.sum(lstm_out * final_attn, dim=1)
        
        # MDN
        pi = torch.softmax(self.pi_head(context), dim=1)
        mu = torch.sigmoid(self.mu_head(context)).view(batch_size, self.num_gaussians, 2)
        sigma_raw = self.sigma_head(context).view(batch_size, self.num_gaussians, 2)
        sigma = torch.sigmoid(sigma_raw) * (MAX_SIGMA - MIN_SIGMA) + MIN_SIGMA
        
        return pi, mu, sigma


def curriculum_hybrid_mdn_loss(pi, mu, sigma, target, epoch, total_epochs):
    """커리큘럼 러닝 + 엔트로피 정규화"""
    batch_size = target.size(0)
    target_expanded = target.unsqueeze(1).expand_as(mu)
    
    # NLL loss
    diff = target_expanded - mu
    log_prob_components = (
        -0.5 * math.log(2 * math.pi) - torch.log(sigma) - 0.5 * (diff / sigma) ** 2
    )
    log_prob = log_prob_components.sum(dim=2)
    weighted_log_prob = log_prob + torch.log(pi + 1e-8)
    nll_loss = -torch.logsumexp(weighted_log_prob, dim=1).mean()
    
    # MSE loss
    pred_mean = (pi.unsqueeze(-1) * mu).sum(dim=1)
    mse_loss = nn.functional.mse_loss(pred_mean, target)
    
    # 커리큘럼 러닝
    if epoch < CURRICULUM_TRANSITION_EPOCH:
        progress = epoch / CURRICULUM_TRANSITION_EPOCH
        mse_weight = CURRICULUM_START_MSE_WEIGHT - (CURRICULUM_START_MSE_WEIGHT - CURRICULUM_END_MSE_WEIGHT) * progress
    else:
        mse_weight = CURRICULUM_END_MSE_WEIGHT
    
    # 엔트로피 정규화
    entropy = 0.5 * (math.log(2 * math.pi * math.e) + 2 * torch.log(sigma))
    avg_entropy = (pi.unsqueeze(-1) * entropy).sum(dim=[1, 2]).mean()
    entropy_penalty = torch.relu(MIN_ENTROPY_TARGET - avg_entropy)
    
    total_loss = nll_loss + mse_weight * mse_loss + ENTROPY_WEIGHT * entropy_penalty
    
    return total_loss, nll_loss, mse_loss, avg_entropy, entropy_penalty, mse_weight


def mdn_predict_improved(pi, mu, sigma, strategy='mean'):
    if strategy == 'mode':
        max_idx = torch.argmax(pi, dim=1)
        pred = mu[torch.arange(len(mu)), max_idx]
    elif strategy == 'mean':
        pred = (pi.unsqueeze(-1) * mu).sum(dim=1)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    
    return torch.clamp(pred, 0.0, 1.0)


# 모델 생성
NGRAM3_VOCAB_SIZE = len(TOP_3GRAMS) + 2  # +2: padding(0), others(21)
NGRAM5_VOCAB_SIZE = len(TOP_5GRAMS) + 2

model = ImprovedMDNPredictor(
    INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS, 
    BIDIRECTIONAL, num_delta_blocks=6, bottleneck_ratio=BOTTLENECK_DIM_RATIO,
    ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
    ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
    ngram_embed_dim=NGRAM_EMBED_DIM
).to(DEVICE)

print("="*70)
print("✅ [V15] N-gram Embedding 모델")
print("="*70)
print(f"핵심 개선:")
print(f"  1. 🔤 3-gram Embedding (vocab: {NGRAM3_VOCAB_SIZE}, dim: {NGRAM_EMBED_DIM})")
print(f"  2. 🔤 5-gram Embedding (vocab: {NGRAM5_VOCAB_SIZE}, dim: {NGRAM_EMBED_DIM})")
print(f"  3. ⚽ type+result 조합으로 전술 패턴 학습")
print(f"  4. 📐 패턴 간 유사도 자동 학습 (임베딩 공간)")
print(f"  5. 📚 커리큘럼 러닝 (MSE {CURRICULUM_START_MSE_WEIGHT} → {CURRICULUM_END_MSE_WEIGHT})")
print(f"  6. 🎲 엔트로피 정규화 (가중치: {ENTROPY_WEIGHT})")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("="*70)

✅ [V15] N-gram Embedding 모델
핵심 개선:
  1. 🔤 3-gram Embedding (vocab: 22, dim: 8)
  2. 🔤 5-gram Embedding (vocab: 22, dim: 8)
  3. ⚽ type+result 조합으로 전술 패턴 학습
  4. 📐 패턴 간 유사도 자동 학습 (임베딩 공간)
  5. 📚 커리큘럼 러닝 (MSE 1.5 → 1.0)
  6. 🎲 엔트로피 정규화 (가중치: 0.5)
  Parameters: 11,522,099


In [36]:
# ======================================================
# [V15] N-gram Embedding 학습
# ======================================================
from torch.optim.lr_scheduler import CosineAnnealingLR

SEEDS = [42, 2024, 777]
PRED_STRATEGY = 'mean'
USE_Y_AUGMENTATION = True

print(f"\n{'='*70}")
print(f"🚀 [V15] N-gram Embedding 학습 시작")
print(f"{'='*70}")
print(f"✅ type+result 조합 패턴으로 전술 상황 명시적 학습")
print(f"✅ 임베딩 공간에서 유사 패턴 자동 학습")
print(f"{'='*70}\n")

all_histories = []

for i, seed in enumerate(SEEDS):
    print(f"\n{'='*70}")
    print(f"📦 [모델 {i+1}/3] Seed {seed}")
    print(f"{'='*70}")
    seed_everything(seed)
    
    train_idx, val_idx = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=seed)
    train_subset_df = train_df[train_df[ID_COL].isin(train_df.groupby(ID_COL).head(1).iloc[train_idx][ID_COL].values)]
    val_subset_df = train_df[train_df[ID_COL].isin(train_df.groupby(ID_COL).head(1).iloc[val_idx][ID_COL].values)]
    
    train_dataset_aug = SoccerDataset(train_subset_df, mode='train', augment_y=USE_Y_AUGMENTATION)
    val_dataset = SoccerDataset(val_subset_df, mode='train', augment_y=False)
    
    print(f"  Train: {len(train_dataset_aug)} episodes")
    print(f"  Val: {len(val_dataset)} episodes")
    
    train_loader = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    model = ImprovedMDNPredictor(
        INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS,
        BIDIRECTIONAL, num_delta_blocks=6, bottleneck_ratio=BOTTLENECK_DIM_RATIO,
        ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
        ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
        ngram_embed_dim=NGRAM_EMBED_DIM
    ).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE - WARMUP_EPOCHS, eta_min=1e-6)
    
    best_dist = float('inf')
    patience_counter = 0
    best_epoch = 0
    
    history = {
        'train_loss': [], 'train_nll': [], 'train_mse': [], 
        'train_entropy': [], 'train_entropy_penalty': [],
        'val_dist': [], 'lr': [], 'mse_weight': []
    }
    
    for epoch in range(EPOCHS_BASE):
        if epoch < WARMUP_EPOCHS:
            for param_group in optimizer.param_groups:
                param_group['lr'] = LR_BASE * (epoch + 1) / WARMUP_EPOCHS
        
        model.train()
        train_loss, train_nll, train_mse = 0.0, 0.0, 0.0
        train_entropy, train_entropy_penalty = 0.0, 0.0
        
        for seqs, ng3, ng5, targets, mask, lens, _ in train_loader:
            seqs = seqs.to(DEVICE)
            ng3 = ng3.to(DEVICE)
            ng5 = ng5.to(DEVICE)
            targets = targets.to(DEVICE)
            mask = mask.to(DEVICE)
            lens = lens.to(DEVICE)
            
            optimizer.zero_grad()
            pi, mu, sigma = model(seqs, ng3, ng5, mask, lens)
            
            loss, nll, mse, entropy, entropy_pen, mse_w = curriculum_hybrid_mdn_loss(
                pi, mu, sigma, targets, epoch, EPOCHS_BASE
            )
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            train_nll += nll.item()
            train_mse += mse.item()
            train_entropy += entropy.item()
            train_entropy_penalty += entropy_pen.item()
        
        model.eval()
        dists = []
        with torch.no_grad():
            for seqs, ng3, ng5, targets, mask, lens, _ in val_loader:
                seqs = seqs.to(DEVICE)
                ng3 = ng3.to(DEVICE)
                ng5 = ng5.to(DEVICE)
                targets = targets.to(DEVICE)
                mask = mask.to(DEVICE)
                lens = lens.to(DEVICE)
                
                pi, mu, sigma = model(seqs, ng3, ng5, mask, lens)
                pred = mdn_predict_improved(pi, mu, sigma, strategy=PRED_STRATEGY)
                p_real = pred.cpu().numpy() * np.array([105.0, 68.0])
                t_real = targets.cpu().numpy() * np.array([105.0, 68.0])
                dists.extend(np.sqrt(np.sum((p_real - t_real)**2, axis=1)))
        
        avg_dist = np.mean(dists)
        avg_loss = train_loss / len(train_loader)
        avg_nll = train_nll / len(train_loader)
        avg_mse = train_mse / len(train_loader)
        avg_entropy = train_entropy / len(train_loader)
        avg_entropy_pen = train_entropy_penalty / len(train_loader)
        
        if epoch >= WARMUP_EPOCHS:
            scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        
        if epoch < CURRICULUM_TRANSITION_EPOCH:
            progress = epoch / CURRICULUM_TRANSITION_EPOCH
            current_mse_weight = CURRICULUM_START_MSE_WEIGHT - (CURRICULUM_START_MSE_WEIGHT - CURRICULUM_END_MSE_WEIGHT) * progress
        else:
            current_mse_weight = CURRICULUM_END_MSE_WEIGHT
        
        history['train_loss'].append(avg_loss)
        history['train_nll'].append(avg_nll)
        history['train_mse'].append(avg_mse)
        history['train_entropy'].append(avg_entropy)
        history['train_entropy_penalty'].append(avg_entropy_pen)
        history['val_dist'].append(avg_dist)
        history['lr'].append(current_lr)
        history['mse_weight'].append(current_mse_weight)
        
        if (epoch + 1) % 5 == 0 or epoch < 5:
            print(f"  [Epoch {epoch+1:2d}/{EPOCHS_BASE}] "
                  f"Loss: {avg_loss:.4f} (NLL: {avg_nll:.4f}, MSE: {avg_mse:.4f}) | "
                  f"Val: {avg_dist:.4f}m | "
                  f"MSE_w: {current_mse_weight:.2f}")
        
        if avg_dist < best_dist:
            best_dist = avg_dist
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), f'v15_ngram_{i}.pth')
            print(f"  ⭐ Best: {best_dist:.4f}m at epoch {best_epoch}")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  🛑 Early Stopping at epoch {epoch+1}")
                print(f"  📌 Best: {best_dist:.4f}m at epoch {best_epoch}")
                break
    
    history['best_dist'] = best_dist
    history['best_epoch'] = best_epoch
    history['seed'] = seed
    all_histories.append(history)
    print(f"  ✅ Best: {best_dist:.4f}m (epoch {best_epoch})")

print(f"\n{'='*70}")
print(f"✅ 학습 완료!")
print(f"{'='*70}")
for i, h in enumerate(all_histories):
    print(f"  Model {i+1}: {h['best_dist']:.4f}m (epoch {h['best_epoch']})")
print(f"  평균 Best: {np.mean([h['best_dist'] for h in all_histories]):.4f}m")
print(f"  최고 성능: {min([h['best_dist'] for h in all_histories]):.4f}m")
print(f"{'='*70}")


🚀 [V15] N-gram Embedding 학습 시작
✅ type+result 조합 패턴으로 전술 상황 명시적 학습
✅ 임베딩 공간에서 유사 패턴 자동 학습


📦 [모델 1/3] Seed 42


Dataset (train): 100%|██████████| 3086/3086 [00:04<00:00, 678.01it/s]


  Train: 24672 episodes
  Val: 3085 episodes
  [Epoch  1/50] Loss: -0.0950 (NLL: -0.5070, MSE: 0.0400) | Val: 16.4524m | MSE_w: 1.50
  ⭐ Best: 16.4524m at epoch 1
  [Epoch  2/50] Loss: -0.2809 (NLL: -0.7832, MSE: 0.0315) | Val: 16.5283m | MSE_w: 1.49
  [Epoch  3/50] Loss: -0.3376 (NLL: -0.9087, MSE: 0.0301) | Val: 15.4662m | MSE_w: 1.48
  ⭐ Best: 15.4662m at epoch 3
  [Epoch  4/50] Loss: -0.3944 (NLL: -1.0161, MSE: 0.0284) | Val: 14.9645m | MSE_w: 1.46
  ⭐ Best: 14.9645m at epoch 4
  [Epoch  5/50] Loss: -0.4459 (NLL: -1.1195, MSE: 0.0272) | Val: 15.3177m | MSE_w: 1.45
  ⭐ Best: 14.6983m at epoch 6
  ⭐ Best: 14.4829m at epoch 8
  [Epoch 10/50] Loss: -0.6578 (NLL: -1.3224, MSE: 0.0243) | Val: 14.5932m | MSE_w: 1.39
  [Epoch 15/50] Loss: -0.8768 (NLL: -1.6012, MSE: 0.0189) | Val: 14.7522m | MSE_w: 1.32


KeyboardInterrupt: 

In [ ]:
# ======================================================
# [V15] N-gram Embedding 추론
# ======================================================
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

models = []
for i in range(3):
    model = ImprovedMDNPredictor(
        INPUT_DIM, HIDDEN_DIM, LSTM_LAYERS, DROPOUT, NUM_GAUSSIANS,
        BIDIRECTIONAL, num_delta_blocks=6, bottleneck_ratio=BOTTLENECK_DIM_RATIO,
        ngram3_vocab_size=NGRAM3_VOCAB_SIZE,
        ngram5_vocab_size=NGRAM5_VOCAB_SIZE,
        ngram_embed_dim=NGRAM_EMBED_DIM
    ).to(DEVICE)
    model.load_state_dict(torch.load(f'v15_ngram_{i}.pth'))
    model.eval()
    models.append(model)

results = []
with torch.no_grad():
    for seqs, ng3, ng5, mask, lens, team_ids, episode_ids in tqdm(test_loader, desc="Inference"):
        seqs = seqs.to(DEVICE)
        ng3 = ng3.to(DEVICE)
        ng5 = ng5.to(DEVICE)
        mask = mask.to(DEVICE)
        lens = lens.to(DEVICE)
        
        preds = []
        for model in models:
            pi, mu, sigma = model(seqs, ng3, ng5, mask, lens)
            pred = mdn_predict_improved(pi, mu, sigma, strategy=PRED_STRATEGY)
            preds.append(pred.cpu().numpy())
        
        avg_pred = np.mean(preds, axis=0)
        for i, eid in enumerate(episode_ids):
            results.append({
                'game_episode': eid, 
                'pred_x': avg_pred[i, 0] * 105.0, 
                'pred_y': avg_pred[i, 1] * 68.0
            })

pred_df = pd.DataFrame(results)
SUBMISSION_PATH = "open_track1/sample_submission.csv"
if os.path.exists(SUBMISSION_PATH):
    sub = pd.read_csv(SUBMISSION_PATH)
else:
    sub = pd.read_csv(TEST_META_PATH).rename(columns={'episode_id': 'game_episode'})[['game_episode']]

final_sub = pd.merge(sub[['game_episode']], pred_df, on='game_episode', how='left')
final_sub = final_sub.rename(columns={'pred_x': 'end_x', 'pred_y': 'end_y'})
if final_sub.isnull().sum().sum() > 0:
    final_sub = final_sub.fillna(50.0)

filename = "v15_ngram_submit.csv"
final_sub.to_csv(filename, index=False)

print(f"\n✅ 제출 파일: {filename}")
print(f"📊 적용된 기법:")
print(f"  1. 🔤 N-gram Embedding (3-gram + 5-gram)")
print(f"  2. ⚽ type+result 조합 패턴 (전술 상황 명시적 학습)")
print(f"  3. 📐 패턴 간 유사도 자동 학습 (임베딩 공간)")
print(f"  4. 📚 커리큘럼 러닝")
print(f"  5. 🎲 엔트로피 정규화")
print(f"  6. 🔒 정보 병목")
print(f"  7. 🌊 Y축 데이터 증강")
print(final_sub.head())

In [ ]:
# 시각화
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('[V14] 14.08m 벽 돌파 - 학습 곡선', fontsize=16, fontweight='bold')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# 1. Validation Distance
ax1 = axes[0, 0]
for i, h in enumerate(all_histories):
    epochs = range(1, len(h['val_dist']) + 1)
    ax1.plot(epochs, h['val_dist'], label=f"Model {i+1}", color=colors[i], linewidth=2)
    best_idx = np.argmin(h['val_dist'])
    ax1.scatter(best_idx + 1, h['val_dist'][best_idx], color=colors[i], s=100, marker='*')
ax1.set_title('Validation Distance')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Distance (m)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. MSE Weight (Curriculum)
ax2 = axes[0, 1]
h = all_histories[0]
epochs = range(1, len(h['mse_weight']) + 1)
ax2.plot(epochs, h['mse_weight'], color='purple', linewidth=2)
ax2.axvline(CURRICULUM_TRANSITION_EPOCH, color='red', linestyle='--', label='Transition')
ax2.set_title('MSE Weight (Curriculum Learning)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Weight')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Entropy
ax3 = axes[1, 0]
for i, h in enumerate(all_histories):
    epochs = range(1, len(h['train_entropy']) + 1)
    ax3.plot(epochs, h['train_entropy'], label=f"Model {i+1}", color=colors[i], linewidth=2)
ax3.axhline(MIN_ENTROPY_TARGET, color='red', linestyle='--', label='Min Target')
ax3.set_title('Entropy (불확실성)')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Entropy')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Entropy Penalty
ax4 = axes[1, 1]
for i, h in enumerate(all_histories):
    epochs = range(1, len(h['train_entropy_penalty']) + 1)
    ax4.plot(epochs, h['train_entropy_penalty'], label=f"Model {i+1}", color=colors[i], linewidth=2)
ax4.set_title('Entropy Penalty (과도한 확신 방지)')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Penalty')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. NLL vs MSE
ax5 = axes[2, 0]
h = all_histories[0]
epochs = range(1, len(h['train_nll']) + 1)
ax5.plot(epochs, h['train_nll'], label='NLL', color='red', linewidth=2)
ax5_twin = ax5.twinx()
ax5_twin.plot(epochs, h['train_mse'], label='MSE', color='blue', linewidth=2)
ax5.set_xlabel('Epoch')
ax5.set_ylabel('NLL', color='red')
ax5_twin.set_ylabel('MSE', color='blue')
ax5.set_title('Loss Components')
ax5.grid(True, alpha=0.3)

# 6. Best Distances
ax6 = axes[2, 1]
best_dists = [h['best_dist'] for h in all_histories]
bars = ax6.bar(range(len(best_dists)), best_dists, color=colors)
ax6.set_xticks(range(len(best_dists)))
ax6.set_xticklabels([f'Model {i+1}' for i in range(len(best_dists))])
ax6.set_ylabel('Best Distance (m)')
ax6.set_title('Best Performance')
for bar, dist in zip(bars, best_dists):
    ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height(), 
             f'{dist:.4f}m', ha='center', va='bottom', fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('v14_breakthrough_history.png', dpi=150)
plt.show()

print(f"\\n최종 결과:")
for i, h in enumerate(all_histories):
    print(f"  Model {i+1}: {h['best_dist']:.4f}m")
print(f"  평균: {np.mean(best_dists):.4f}m")
print(f"\\n🎯 14.08m 벽 돌파 전략:")
print(f"  1. 초기 MSE 강조로 대략적 위치 빠르게 학습")
print(f"  2. 후기 NLL 강조로 확률 분포 정교화")
print(f"  3. 엔트로피 제약으로 과도한 확신 방지 → 일반화 향상")
print(f"  4. 정보 병목으로 노이즈 필터링 → 과적합 방지")